# Earlier data-preparation experiments

This notebook keeps the earlier attempts to create a patient-based subset, assign validation folds and inspect missing metadata. Its question was whether the full training dataset could be divided while keeping each patient's images together.

It predates the current numbered notebooks. There is no previous notebook in the active workflow that feeds into it. The current cleaning, exploration and patient-splitting notebooks replace these experiments; use those notebooks for new work.

Keep this file as a record of how the approach developed. The code mixes older folder paths and writes a fold file, so reading the saved outputs is safer than running it as part of the current workflow.

## Try a patient-based subset

The first cell takes one of ten groups as a smaller dataset. It groups images by patient so the same person's images stay together. The folder path belongs to an earlier checkout.

In [ ]:
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

# Load metadata
train_df = pd.read_csv("C:/Users/thoma/Documents/Monash/ADS2002/Melanoma-Detection/data/raw/train.csv")

sgkf = StratifiedGroupKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

# We only need one of the 10 folds
for _, subset_idx in sgkf.split(
    X=train_df,
    y=train_df["target"],
    groups=train_df["patient_id"]
):
    subset_df = train_df.iloc[subset_idx].copy()
    break

## Try five validation folds

The next cell assigns the full dataset to five folds and writes a CSV. Read the table's `count` as the number of images, `sum` as the number of malignant images and `mean` as their fraction in that fold. Similar fractions show similar class balance; they do not establish model performance.

In [2]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

DATA_DIR = Path("../data").resolve()
RAW_DIR = DATA_DIR / "raw"

df = pd.read_csv("C:/Users/thoma/Documents/GitHub/ADS2002-Melanoma/data/raw/train.csv")

splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

df["fold"] = -1

for fold, (_, val_idx) in enumerate(
    splitter.split(
        X=df,
        y=df["target"],
        groups=df["patient_id"],
    )
):
    df.loc[val_idx, "fold"] = fold

# Check class balance in each fold
print(
    df.groupby("fold")["target"]
      .agg(["count", "sum", "mean"])
)

output_dir = DATA_DIR / "processed"
output_dir.mkdir(exist_ok=True)

df.to_csv(output_dir / "train_folds.csv", index=False)

      count  sum      mean
fold                      
0      6627  117  0.017655
1      6625  116  0.017509
2      6621  116  0.017520
3      6628  117  0.017652
4      6625  118  0.017811


## Inspect the subset size and class balance

The printed percentages compare the malignant fraction in the full dataset and the subset. The patient counts help show how many separate people contribute those images. There is no graph because these cells report a few exact counts and proportions.

In [2]:
print("Original:", len(train_df))
print("Subset:", len(subset_df))
print("Percentage:", len(subset_df) / len(train_df) * 100)

print("\nOriginal target distribution:")
print(train_df["target"].value_counts(normalize=True))

print("\nSubset target distribution:")
print(subset_df["target"].value_counts(normalize=True))

print("\nPatients in original:", train_df["patient_id"].nunique())
print("Patients in subset:", subset_df["patient_id"].nunique())

Original: 33126
Subset: 3568
Percentage: 10.770995592585884

Original target distribution:
target
0    0.98237
1    0.01763
Name: proportion, dtype: float64

Subset target distribution:
target
0    0.983744
1    0.016256
Name: proportion, dtype: float64

Patients in original: 2056
Patients in subset: 207


## Count missing sex values

This cell loads another older fold file and counts missing values. Its saved count relates to that file, not necessarily the current d1 development data.

In [8]:
df = pd.read_csv("C:/Users/thoma/Documents/Monash/ADS2002/Melanoma-Detection/data/processed/train_folds.csv")
df["sex"].isna().sum()

65

## Try an explicit category for missing sex

Replacing missing sex values with `Unknown` keeps those rows available for later modelling. A zero count after replacement means the missing values now have a label; it does not mean the original sex information was recovered.

In [15]:
df["sex"] = df["sex"].fillna("Unknown")
df["sex"].isna().sum()

0

## Count missing anatomical sites

The output reports how many rows have no site value. This cell only counts them; it does not resolve the missing information.

In [12]:
df["anatom_site_general_challenge"].isna().sum()

527

## Count missing ages

The output reports how many rows have no age value. Any later method for filling missing ages must learn its replacement value from training patients only.

In [13]:
df["age_approx"].isna().sum()

68

## Limitations

This is a set of early experiments rather than a complete run from preparation to evaluation. It reads files from different older paths, and its saved outputs do not establish that all cells used the same data or were run in order.

The saved subset has 3,568 images from 207 patients, which differs from the current d1 development dataset. The missing-value counts also refer to an older file. No model is fitted or evaluated here, so class balance cannot show how well a classifier would perform. Running the code can write a fold file, which is another reason to keep it outside the active workflow.

## Summary

These experiments showed how patient grouping could be used to create a smaller subset and validation folds. They also identified missing values in sex, site and age in the older files.

Retain this notebook as an archive. There is no trained model to retain, and its outputs should not be used as current project results. For new work, begin with the matching cleaning notebook, then exploration and patient splitting in the numbered workflow.